# GCP Pipeline — Same Queries, Infinite Scale

**The Analytics Pipeline notebook ran on Pandas. This one runs on GCS + BigQuery.**

Same data. Same questions. But BigQuery handles 50 million rows without breaking a sweat.

This notebook is meant to be run cell-by-cell in a terminal or Cloud Shell — the `!` prefix runs shell commands from Jupyter/Colab.

---

## Step 0: GCP Setup (One-Time)

Run these once. Skip if already done.

### Option A: CLI (this notebook)
Run the cells below.

### Option B: GCP Console UI
1. Go to [console.cloud.google.com](https://console.cloud.google.com)
2. Select your project from the dropdown at the top (or create one: **Select Project → New Project → name it → Create**)
3. **Enable APIs:** Hamburger menu → **APIs & Services → Library** → search "BigQuery API" → **Enable**. Repeat for "Cloud Storage"
4. **Create a bucket:** Hamburger menu → **Cloud Storage → Buckets → Create** → name it `de-accelerator-data-pipeline-project-490820` → location: `us-central1` → **Create**
5. **Upload files:** Click into the bucket → **Upload Files** → select `campaigns.csv`, `products.csv`, `orders.csv`, `calls.json` → upload into a folder called `bronze/` (create the folder first: **Create Folder → `bronze`**)
6. **Create BigQuery dataset:** Hamburger menu → **BigQuery** → click the three dots next to your project name → **Create Dataset** → name: `call_center_raw` → location: `us-central1` → **Create**
7. **Load tables:** Click `call_center_raw` dataset → **Create Table** → Source: Google Cloud Storage → browse to `gs://de-accelerator-.../bronze/campaigns.csv` → Table name: `campaigns` → Auto-detect schema: checked → **Create Table**. Repeat for each file.

In [ ]:
# === CONFIGURATION — Change these ===
PROJECT_ID = "data-pipeline-project-490820"
BUCKET_NAME = f"de-accelerator-{PROJECT_ID}"  # Globally unique
DATASET_NAME = "call_center_raw"
LOCATION = "us-central1"

print(f"Project:  {PROJECT_ID}")
print(f"Bucket:   gs://{BUCKET_NAME}/")
print(f"Dataset:  {DATASET_NAME}")
print(f"Location: {LOCATION}")

### Installing the Google Cloud CLI

**On Colab:** Already installed. Just authenticate:
```python
from google.colab import auth
auth.authenticate_user()
```

**On Mac (Homebrew):**
```bash
brew install --cask google-cloud-sdk
gcloud init    # login + select project
```

**On Mac (manual):**
```bash
curl https://sdk.cloud.google.com | bash
exec -l $SHELL    # restart shell to pick up the new PATH
gcloud init
```

**On Windows:**
1. Download installer: [cloud.google.com/sdk/docs/install](https://cloud.google.com/sdk/docs/install) → click Windows → run the `.exe`
2. Follow prompts — it adds `gcloud` to PATH automatically
3. Open a new terminal and run: `gcloud init`

**Verify installation:**
```bash
gcloud --version
bq --version
```

Both `gcloud` (general GCP CLI) and `bq` (BigQuery CLI) are included in the same install.

In [ ]:
# Authenticate
# On Colab: this pops up a Google login window
# On local: uncomment the gcloud auth login line instead

import os
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated via Colab.")
else:
    # Running locally — uncomment the next line if you haven't logged in yet:
    # !gcloud auth login
    print("Running locally. Use 'gcloud auth login' if not authenticated.")

# Set the project
!gcloud config set project {PROJECT_ID}

In [ ]:
# Enable required APIs (only needed once per project)
!gcloud services enable bigquery.googleapis.com
!gcloud services enable storage.googleapis.com
print("APIs enabled.")

# Check if billing is linked (required for GCS and BigQuery)
!gcloud billing projects describe {PROJECT_ID} --format="value(billingAccountName)" 2>&1

### Billing Setup (Required)

GCS and BigQuery require a billing account linked to the project — even for the free tier. Without billing, bucket creation and queries will fail with `403: billing account disabled`.

**Free tier covers everything in this notebook:**
- GCS: 5 GB storage free
- BigQuery: 1 TB of queries free per month
- Our dataset is ~1 MB. Cost: effectively $0.

**Option A: Console UI**
1. Go to [console.cloud.google.com/billing](https://console.cloud.google.com/billing)
2. If no billing account exists: **Create Account** → add a credit card (will not be charged for this volume)
3. Go to [console.cloud.google.com/billing/linkedaccount](https://console.cloud.google.com/billing/linkedaccount?project=data-pipeline-project-490820)
4. **Link a billing account** → select your account → **Set Account**

**Option B: CLI**

In [ ]:
# List your billing accounts
!gcloud billing accounts list

# If you see a billing account, link it to the project:
# Replace XXXXXX-XXXXXX-XXXXXX with your billing account ID from the output above
# !gcloud billing projects link {PROJECT_ID} --billing-account=XXXXXX-XXXXXX-XXXXXX

# Verify billing is linked
!gcloud billing projects describe {PROJECT_ID} --format="value(billingEnabled)"
# Should print: True

---

## IAM (Identity and Access Management) — Who Can Do What

**IAM (pronounced "I-am")** controls who has access to which GCP resources. Every action in GCP — uploading a file, running a query, creating a bucket — requires permission.

### Key Concepts

| Concept | What It Means | Analogy | AWS Equivalent |
|:---|:---|:---|:---|
| **Principal** (Member) | Who is accessing — a user email, a service account, or a group | The person showing their badge at the door | IAM User / Role / Group |
| **Role** | A bundle of permissions (e.g., `roles/bigquery.dataEditor` = can read + write BigQuery tables) | The badge type — "employee" vs "visitor" vs "admin" | IAM Policy document (the permissions list) |
| **Policy** | A JSON document that binds principals to roles on a specific resource. Every resource (project, bucket, dataset) has a policy attached. | The access control list ON the door — "these people with these badges can enter this room" | IAM Policy attached to a resource |
| **Binding** | One entry in a policy: one role + one or more principals | One line on the access list: "BigQuery Editor: alice@, bob@, pipeline-sa@" | One Statement in an IAM Policy |
| **Service Account** | A non-human identity for applications (e.g., your pipeline runs as a service account, not as your personal email) | A robot with its own badge — the pipeline authenticates as this identity | IAM Role (assumed by Lambda/EC2) |

### How a Policy Looks

Every GCP resource has a policy. The policy is a JSON document:

```json
{
  "bindings": [
    {
      "role": "roles/bigquery.dataEditor",
      "members": [
        "user:analyst@company.com",
        "serviceAccount:pipeline@project.iam.gserviceaccount.com"
      ]
    },
    {
      "role": "roles/storage.objectViewer",
      "members": [
        "user:analyst@company.com"
      ]
    }
  ]
}
```

This policy says: "analyst@ can edit BigQuery tables AND view GCS objects. The pipeline service account can edit BigQuery tables."

### Key Difference from AWS

In **AWS**, policies are attached to **users/roles** — "this user can do these things."

In **GCP**, policies are attached to **resources** — "this project/bucket/dataset allows these people to do these things."

The question flips:
- AWS: "What can this user do?" → look at the user's attached policies
- GCP: "Who can access this resource?" → look at the resource's policy

### Common Roles for Data Engineering

| Role | Permissions | When to Use |
|:---|:---|:---|
| `roles/bigquery.dataViewer` | Read tables and views | Analysts who only query, never modify |
| `roles/bigquery.dataEditor` | Read + write tables | Pipelines that load/transform data |
| `roles/bigquery.admin` | Full BigQuery control (create/delete datasets, manage access) | Project owner, admin |
| `roles/storage.objectViewer` | Read files in GCS | Applications that read data from buckets |
| `roles/storage.objectCreator` | Upload files to GCS | Pipelines that write data |
| `roles/storage.admin` | Full GCS control | Project owner, admin |

### Console UI: Check Your Permissions
1. **Hamburger menu → IAM & Admin → IAM**
2. Find your email in the list — you'll see which roles (bindings) you have
3. Click **View by: Principals** to see all users and their roles
4. Click **View by: Roles** to see all roles and who has them
5. You need at least `BigQuery Data Editor` + `Storage Object Admin` for this notebook

### Console UI: View a Resource's Policy
1. **BigQuery → click a dataset → Share** — shows who has access to that dataset
2. **Cloud Storage → click a bucket → Permissions** — shows the bucket's policy
3. Each resource has its own policy — project-level roles inherit down to all resources in the project

### CLI: Check and Grant Permissions

In [ ]:
# Check who you are authenticated as
!gcloud auth list

# Check your current IAM policy (who has what role on this project)
# You will see a mix of:
#   - YOUR accounts (user:your-email@gmail.com) — you created these
#   - AUTO-CREATED service accounts — GCP creates these when you enable APIs:
#       - 123456-compute@developer... → Default Compute Engine SA (auto when Compute API enabled)
#       - 123456@cloudservices...     → Google APIs SA (manages internal GCP operations)
#       - service-123456@...          → Service agents for Dataflow, Composer, etc. (auto)
#   - SERVICE ACCOUNTS YOU CREATED — e.g., dataflow-demo-sa@ (you created this for a specific purpose)
#
# The number 123456 is your project number (not project ID) — GCP uses it internally.
# You don't need to manage the auto-created ones. They handle GCP's internal plumbing.

!gcloud projects get-iam-policy {PROJECT_ID} --flatten="bindings[].members" --format="table(bindings.role, bindings.members)" 2>&1 | head -30

# Grant a new user access (e.g., a student or team member)
# They need BigQuery Editor (to create tables + run queries) + Storage Admin (to upload files)
# Uncomment and replace EMAIL:
# USER_EMAIL = "student@example.com"
# !gcloud projects add-iam-policy-binding {PROJECT_ID} --member="user:{USER_EMAIL}" --role="roles/bigquery.dataEditor"
# !gcloud projects add-iam-policy-binding {PROJECT_ID} --member="user:{USER_EMAIL}" --role="roles/storage.objectAdmin"
# print(f"Granted BigQuery Editor + Storage Admin to {USER_EMAIL}")

# Console UI way:
# IAM & Admin → IAM → "+ Grant Access" → enter their email → 
# Select roles: "BigQuery Data Editor" + "Storage Object Admin" → Save

In [ ]:
# Create a storage bucket
# Bucket names must be globally unique across all of GCP
!gcloud storage buckets create gs://{BUCKET_NAME}/ --location={LOCATION} 2>&1 || echo "Bucket may already exist — that is fine."

# Verify
!gcloud storage ls gs://{BUCKET_NAME}/ 2>&1 | head -5
print(f"\nBucket ready: gs://{BUCKET_NAME}/")

---

## Step 1: BRONZE — Upload Raw Data to GCS

This is the cloud equivalent of "loading a CSV file." Instead of reading from your laptop's disk, the data lives in GCS (Google Cloud Storage) — accessible from anywhere, scalable to petabytes.

In production, data would land here automatically (API writes, file drops, streaming). For now, we upload manually.

### Console UI Way
1. Go to [console.cloud.google.com/storage/browser](https://console.cloud.google.com/storage/browser?project=data-pipeline-project-490820)
2. Click your bucket (`de-accelerator-data-pipeline-project-490820`)
3. Click **Create Folder** → name it `bronze` → **Create**
4. Click into the `bronze` folder
5. Click **Upload Files** → select `campaigns.csv`, `products.csv`, `orders.csv`, `calls.json` from your local machine
6. Wait for upload to complete — you should see all 4 files listed

> **Download the data files first:** Go to [github.com/sunilmogadati/systems-in-production/tree/main/data](https://github.com/sunilmogadati/systems-in-production/tree/main/data) → click each file → **Download raw file** (or clone the repo)

### CLI Way (cells below)

In [ ]:
import os

# On Colab: clone the repo to get the data files
if not os.path.exists("../data") and not os.path.exists("data"):
    print("Downloading data from GitHub...")
    !git clone --depth 1 https://github.com/sunilmogadati/systems-in-production.git /tmp/de-repo 2>/dev/null
    DATA_DIR = "/tmp/de-repo/data"
elif os.path.exists("../data/calls.json"):
    DATA_DIR = "../data"
elif os.path.exists("data/calls.json"):
    DATA_DIR = "data"
else:
    DATA_DIR = os.path.expanduser("~/Downloads/systems-in-production/data")

print(f"Data source: {DATA_DIR}")
print(f"Files: {os.listdir(DATA_DIR)}")

In [ ]:
# Upload all data files to the Bronze layer
# Bronze = raw, untouched. Same principle as the Pandas notebook.

!gcloud storage cp {DATA_DIR}/campaigns.csv gs://{BUCKET_NAME}/bronze/
!gcloud storage cp {DATA_DIR}/products.csv gs://{BUCKET_NAME}/bronze/
!gcloud storage cp {DATA_DIR}/orders.csv gs://{BUCKET_NAME}/bronze/
!gcloud storage cp {DATA_DIR}/calls.json gs://{BUCKET_NAME}/bronze/

print("\nBronze layer loaded.")

In [ ]:
# Verify — list everything in the bronze folder
!gcloud storage ls gs://{BUCKET_NAME}/bronze/

---

## Step 2: Load into BigQuery

BigQuery is Google's serverless data warehouse. No servers to manage. You pay per query (bytes scanned). It can scan terabytes in seconds.

This is the cloud equivalent of `pd.read_csv()` — but it scales to billions of rows.

### Console UI Way
1. Go to [console.cloud.google.com/bigquery](https://console.cloud.google.com/bigquery?project=data-pipeline-project-490820)
2. **Create dataset:** In the left panel, click the three dots (⋮) next to your project name → **Create Dataset** → Dataset ID: `call_center_raw` → Location: `us-central1` → **Create Dataset**
3. **Load campaigns table:**
   - Click `call_center_raw` in the left panel → **Create Table**
   - Source: **Google Cloud Storage**
   - Browse to: `gs://de-accelerator-data-pipeline-project-490820/bronze/campaigns.csv`
   - File format: **CSV**
   - Table name: `campaigns`
   - Check **Auto detect** (under Schema)
   - Click **Create Table**
4. **Repeat for each file:**
   - `products.csv` → table name: `products`
   - `orders.csv` → table name: `orders`
   - `calls.json` → table name: `calls` (change file format to **JSON (Newline delimited)**)
5. **Verify:** Click each table → **Preview** tab to see the data. Click **Schema** tab to see column names and types.

### CLI Way (cells below)

In [ ]:
# Create a BigQuery dataset (like a schema/database — a container for tables)
!bq mk --dataset --location={LOCATION} {PROJECT_ID}:{DATASET_NAME} 2>&1 || echo "Dataset may already exist."

print(f"Dataset ready: {PROJECT_ID}:{DATASET_NAME}")

In [ ]:
# Load CSV files into BigQuery tables
# --autodetect lets BigQuery infer column names and types from the CSV header

!bq load --autodetect --replace --source_format=CSV \
  {DATASET_NAME}.campaigns \
  gs://{BUCKET_NAME}/bronze/campaigns.csv

print("campaigns loaded.")

In [ ]:
!bq load --autodetect --replace --source_format=CSV \
  {DATASET_NAME}.products \
  gs://{BUCKET_NAME}/bronze/products.csv

print("products loaded.")

In [ ]:
!bq load --autodetect --replace --source_format=CSV \
  {DATASET_NAME}.orders \
  gs://{BUCKET_NAME}/bronze/orders.csv

print("orders loaded.")

In [ ]:
# Load the calls JSON
# calls.json is newline-delimited JSON (NDJSON) — each line is one record
!bq load --autodetect --replace --source_format=NEWLINE_DELIMITED_JSON \
  {DATASET_NAME}.calls \
  gs://{BUCKET_NAME}/bronze/calls.json

print("calls loaded.")

In [ ]:
# Verify — list all tables in the dataset
!bq ls {DATASET_NAME}

In [ ]:
# Preview the calls table — does it look right?
!bq head -n 3 {DATASET_NAME}.calls

---

## Step 3: Query BigQuery — Same Questions, Cloud Scale

These are the **same queries** from the Analytics Pipeline notebook. There, they ran on Pandas. Here, they run on BigQuery SQL.

On 500 rows, both are fast. On 50 million rows, Pandas crashes. BigQuery finishes in seconds.

### Console UI Way — Running Queries
1. Go to [console.cloud.google.com/bigquery](https://console.cloud.google.com/bigquery?project=data-pipeline-project-490820)
2. In the left panel, expand your project → `call_center_raw` → click any table → **Preview** tab to see the data
3. Click **+ Compose New Query** (button near top, or the `+` tab in the editor area)
4. Paste any SQL query from the cells below into the editor
5. **Important:** Use the full table path in the console. Replace table references like this:
   - `{T}.calls` → `` `data-pipeline-project-490820.call_center_raw.calls` ``
   - `{T}.campaigns` → `` `data-pipeline-project-490820.call_center_raw.campaigns` ``
6. Click **Run** → results appear at the bottom
7. Click **Save Results** → download as CSV, save as Google Sheets, or save as a new BigQuery table

### Console UI Way — Exploring Tables
- Click any table in the left panel → **Schema** tab shows column names, types, and descriptions
- **Preview** tab shows sample rows without running a query (free — no bytes scanned)
- **Details** tab shows row count, size, creation date, last modified

### CLI Way (cells below)
Run the helper cell first (defines `run_bq` and `T`), then run any query cell.

In [ ]:
# Helper function — runs a BigQuery SQL query and prints results
# WHY: Colab's ! shell commands don't expand Python variables inside single quotes.
# This function builds the query string in Python and calls bq via subprocess.

import subprocess

def run_bq(sql):
    """Run a BigQuery SQL query and print the results."""
    result = subprocess.run(["bq", "query", "--use_legacy_sql=false", sql],
                            capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("ERROR:", result.stderr)

# Shorthand for fully qualified table names
T = f"{PROJECT_ID}.{DATASET_NAME}"
print(f"Table prefix: {T}  (e.g., {T}.calls)")

In [ ]:
# Query 1: Call count and average duration by campaign

run_bq(f"""
SELECT
    c.campaign_name,
    c.channel,
    COUNT(*) AS total_calls,
    ROUND(AVG(calls.duration), 1) AS avg_duration
FROM `{T}.calls` AS calls
JOIN `{T}.campaigns` AS c
    ON CAST(calls.dnis AS STRING) = CAST(c.dnis AS STRING)
GROUP BY c.campaign_name, c.channel
ORDER BY avg_duration DESC
""")

In [ ]:
# Query 2: Disposition (outcome) distribution

run_bq(f"""
SELECT
    disposition,
    COUNT(*) AS count
FROM `{T}.calls`
GROUP BY disposition
ORDER BY count DESC
""")

In [ ]:
# Query 3: Calls by day of week

run_bq(f"""
SELECT
    FORMAT_DATE('%A', date) AS day_name,
    CASE WHEN EXTRACT(DAYOFWEEK FROM date) IN (1, 7) THEN true ELSE false END AS is_weekend,
    COUNT(*) AS total_calls
FROM `{T}.calls`
WHERE date IS NOT NULL
GROUP BY day_name, is_weekend
ORDER BY total_calls DESC
""")

In [ ]:
# Query 4: VA vs Live channel comparison

run_bq(f"""
SELECT
    c.channel,
    COUNT(*) AS total_calls,
    ROUND(AVG(calls.duration), 1) AS avg_duration
FROM `{T}.calls` AS calls
JOIN `{T}.campaigns` AS c
    ON CAST(calls.dnis AS STRING) = CAST(c.dnis AS STRING)
GROUP BY c.channel
""")

In [ ]:
# Query 5: Calls by hour of day

run_bq(f"""
SELECT
    EXTRACT(HOUR FROM start_time) AS hour_of_day,
    COUNT(*) AS total_calls,
    ROUND(AVG(duration), 1) AS avg_duration
FROM `{T}.calls`
WHERE start_time IS NOT NULL
GROUP BY hour_of_day
ORDER BY hour_of_day
""")

In [ ]:
# Query 6: Orders and revenue by campaign

run_bq(f"""
SELECT
    c.campaign_name,
    COUNT(DISTINCT o.order_id) AS order_count,
    ROUND(SUM(o.total), 2) AS total_revenue,
    ROUND(AVG(o.total), 2) AS avg_order_value
FROM `{T}.orders` AS o
JOIN `{T}.calls` AS calls
    ON o.call_id = calls.call_id
JOIN `{T}.campaigns` AS c
    ON CAST(calls.dnis AS STRING) = CAST(c.dnis AS STRING)
GROUP BY c.campaign_name
ORDER BY total_revenue DESC
""")

---

## What Just Happened — Pandas vs BigQuery

| | Pandas (Analytics_Pipeline notebook) | BigQuery (this notebook) |
|---|---|---|
| **Data location** | Your laptop's RAM | Google Cloud — distributed across servers |
| **Query language** | Python (`.groupby().agg()`) | SQL |
| **500 rows** | 2 seconds | 2 seconds |
| **50 million rows** | Crashes (out of memory) | 3-5 seconds |
| **500 million rows** | Impossible | 10-15 seconds |
| **Cost** | Free (your hardware) | ~$0.005 per query on 500 rows. ~$2.50 per query on 50M rows. |
| **Scheduling** | Run manually | Cloud Composer (Airflow) runs it every night |
| **Collaboration** | Share the notebook | Anyone with BigQuery access runs the same queries |

**The pipeline pattern is identical.** Bronze → Silver → Gold → Queries. Only the execution engine changed.

---

## Next Steps

1. **Build the star schema in BigQuery** — `CREATE TABLE` with surrogate keys, dimension tables, partitioning
2. **Schedule with Cloud Composer** — DAG that runs Bronze → Silver → Gold every night
3. **Add data quality checks** — detect duplicates, nulls, timezone issues before they reach Gold
4. **Connect a dashboard** — Looker Studio or Data Studio reads from the Gold tables

In [ ]:
# Cleanup — uncomment to delete resources when done
# !bq rm -r -f {DATASET_NAME}
# !gcloud storage rm -r gs://{BUCKET_NAME}/
# print("Resources deleted.")